# Fine-Tune SLM on Cocos2d-x JS/TS Training Data

This notebook fine-tunes a small language model on synthetic Cocos2d-x coding data.

**Requirements:** GPU runtime (T4 or better). Go to Runtime → Change runtime type → GPU.

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl wandb

## 2. Clone Repo & Load Data

In [ ]:
import os

# Clone the repo (change to your username if forked)
REPO = "nmnhut-it/fine-tune-cocoos
if not os.path.exists("fine-tune-cocoos"):
    !git clone https://github.com/{REPO}.git

os.chdir("fine-tune-cocoos")

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files="data/train.jsonl", split="train")
test_ds = load_dataset("json", data_files="data/test.jsonl", split="train")

print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")
print(train_ds[0])

## 3. Choose Base Model

Pick an SLM. Good options:
- `microsoft/phi-2` (2.7B)
- `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (1.1B)
- `Qwen/Qwen2.5-Coder-1.5B-Instruct` (1.5B)
- `google/codegemma-2b` (2B)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # Change as needed

## 4. Load Model with QLoRA (4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Format Data as Alpaca Prompts

In [ ]:
ALPACA_TEMPLATE = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""

MAX_LENGTH = 2048

def format_and_tokenize(example):
    text = ALPACA_TEMPLATE.format(
        instruction=example["instruction"],
        output=example["output"],
    )
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_tokenized = train_ds.map(format_and_tokenize, remove_columns=train_ds.column_names)
test_tokenized = test_ds.map(format_and_tokenize, remove_columns=test_ds.column_names)

print(f"Train tokens shape: {len(train_tokenized)}, Test tokens shape: {len(test_tokenized)}")

## 6. Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",  # Change to "wandb" if you want W&B logging
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

## 7. Save LoRA Adapter

In [ ]:
ADAPTER_PATH = "./cocos2dx-lora-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")

## 8. Test Inference

In [ ]:
def generate(instruction, max_new_tokens=512):
    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Test with examples from the test set
for i in [0, 25, 50, 75]:
    example = test_ds[i]
    print(f"\n{'='*60}")
    print(f"INSTRUCTION: {example['instruction'][:200]}...")
    print(f"\nGENERATED:\n{generate(example['instruction'])}")
    print(f"\nEXPECTED:\n{example['output'][:300]}...")

## 9. Evaluate on Test Set

In [ ]:
eval_results = trainer.evaluate()
print(f"Test Loss: {eval_results['eval_loss']:.4f}")
print(f"Test Perplexity: {2**eval_results['eval_loss']:.2f}")

## 10. (Optional) Push Adapter to Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub("your-username/cocos2dx-coder-lora")
# tokenizer.push_to_hub("your-username/cocos2dx-coder-lora")